In [1]:
# conda install -c conda-forge pandas numpy matplotlib plotly pytorch seaborn darts sktime scikit-learn -y
# pip install datasets transformers chronos-forecasting darts torch_geometric

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import polars as pl
import random
import json
import pickle

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader , Dataset , random_split
from torchmetrics.regression import MeanAbsoluteError
import lightning as L
import math

# from darts.ad.anomaly_model import ForecastingAnomalyModel
# from darts.ad.scorers import ResidualsNormScorer
# from darts.models import RegressionEnsembleModel, NaiveSeasonal, LinearRegressionModel
# from darts.utils.data import PastCovariatesSequentialDataset
# from darts import TimeSeries

from torch_geometric.nn import GCNConv
from torch_geometric.data import Dataset, InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from sklearn.metrics.pairwise import haversine_distances,euclidean_distances

In [30]:
train_df = pd.read_parquet('/project/ai901504-ai0004/501641_Big/week6/test_df_normal.parquet')

In [31]:
train_df

# Model

### LightGBM classifier

#### Preprocess

In [ ]:
def create_lag(df):
    

#### Classifier

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.metrics import roc_curve

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

def autotrain(df, test_df):
    # df = df.reset_index()
    # test_df = test_df.reset_index()
    model_save_path="/project/ai901504-ai0004/501641_Big/week6/Classifier_prob_model"
    os.makedirs(model_save_path, exist_ok=True)

    items = df['Station'].unique()

    # df['hour'] = df['timestamp'].dt.hour
    # df['day_of_month'] = df['timestamp'].dt.day
    # df['month'] = df['timestamp'].dt.month

    # test_df['hour'] = test_df['timestamp'].dt.hour
    # test_df['day_of_month'] = test_df['timestamp'].dt.day
    # test_df['month'] = test_df['timestamp'].dt.month

    base_features = ['Temperature', 'Pressure', 'Humidity', 'GSMap', 'long', 'lat', 'month', 'day_of_month', 'hour']
    categorical_features = ['month', 'day_of_month', 'hour']

    # Create binary target
    df['classifier'] = (df['HII'] > 0).astype(int)
    test_df['classifier'] = (test_df['HII'] > 0).astype(int)

    for item in items:
        print(f"\nProcessing Station {item}...")
        item_df = df[df['item_id'] == item].copy()
        item_test_df = test_df[test_df['item_id'] == item].copy()

        current_features = base_features

        # Train classifier
        print(f"  Training classifier for item {item}...")
        X_train_clf = item_df[current_features]
        y_train_clf = item_df['classifier']

        X_val_clf = item_test_df[current_features]
        y_val_clf = item_test_df['classifier']

        classifier = LGBMClassifier(
            objective='binary',
            metric='binary_logloss',
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=64,
            max_depth=10,
            verbose=-1,
            n_jobs=-1,
            random_state=42
        )

        classifier.fit(
            X_train_clf, y_train_clf,
            eval_set=[(X_val_clf, y_val_clf)],
            eval_metric='logloss',
            categorical_feature=categorical_features,
            callbacks=[lgb.early_stopping(10, verbose=False)]
        )

        val_probs = classifier.predict_proba(X_val_clf)[:, 1]

        clf_logloss = log_loss(y_val_clf, val_probs)
        clf_auc = roc_auc_score(y_val_clf, val_probs)
        
        print(f"  Item {item} Classifier LogLoss: {clf_logloss:.4f}")
        print(f"  Item {item} Classifier AUC: {clf_auc:.4f}")

        fpr, tpr, thresholds = roc_curve(y_val_clf, val_probs)
        optimal_threshold = thresholds[np.argmax(tpr - fpr)]
        print(f"  Item {item} Optimal Threshold: {optimal_threshold:.4f}")

        importance_df = pd.DataFrame({
                'feature': current_features,
                'importance': classifier.feature_importances_
            }).sort_values('importance', ascending=False)

        top5 = importance_df.head(5)
        print(f"  Top 5 important features for item {item}:")
        print(top5.to_string(index=False))

        #Save model
        model_filename = f"{item}.pkl"
        model_path = os.path.join(model_save_path, model_filename)
        
        with open(model_path, 'wb') as f:
            pickle.dump(classifier, f)
        
        # Save model metadata
        metadata = {
            'item_id': item,
            'threshold': optimal_threshold,
            'features': current_features,
            'logloss': clf_logloss,
            'auc': clf_auc,
            'model_filename': model_filename,
            'training_records': len(item_df),
            'train_size': len(X_train_clf),
            'val_size': len(X_val_clf)
        }
        
        metadata_filename = f"{item}_metadata.json"
        metadata_path = os.path.join(model_save_path, metadata_filename)
        
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"  Saved model: {model_filename}")
        print(f"  Saved metadata: {metadata_filename}")


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from sklearn.metrics import roc_curve


#input only dataframe that index are timestamp
def classifier_probability_for_autogluon(df, model_path='/project/ai901504-ai0004/501641_Big/week6/Classifier_prob_model/',
                           metadata_path='/project/ai901504-ai0004/501641_Big/week6/Classifier_prob_model/'):
    #Change to right format
    # df = df.reset_index()

    df['row_index'] = np.arange(len(df))

    items = df['Station'].unique()

    # df['hour'] = df['timestamp'].dt.hour
    # df['day_of_month'] = df['timestamp'].dt.day
    # df['month'] = df['timestamp'].dt.month

    #Feature from df
    base_features = ['Temperature', 'Pressure', 'Humidity', 'GSMap', 'long', 'lat', 'month', 'day_of_month', 'hour']

    #Looping
    for item in items:
        print(f"\nProcessing Station {item}...")
        item_df = df[df['item_id'] == item].copy()

        #Load model
        model_path_item = os.path.join(model_path, f"{item}.pkl")
        metadata_path_item = os.path.join(metadata_path, f"{item}_metadata.json")

        with open(model_path_item, 'rb') as f:
            model = pickle.load(f)

        with open(metadata_path_item, 'r') as f:
            metadata = json.load(f)

        print(f"Loaded model for item {item}.")
        print(f"Optimal threshold from training: {metadata['threshold']:.4f}")

        X_predict = item_df[base_features]

        #Use
        
        # Optimal Threshold -> fpr, tpr, thresholds = roc_curve(y_val_clf, val_probs) -> optimal_threshold = thresholds[np.argmax(tpr - fpr)]
        # Use for prediction rain or no rain
        probabilities = model.predict_proba(X_predict)[:, 1]
        threshold = metadata['threshold']
        predictions = (probabilities >= threshold).astype(int)

        df.loc[item_df['row_index'], 'probability'] = probabilities
        df.loc[item_df['row_index'], 'prediction'] = predictions

        print(f"  Updated {len(probabilities)} rows for {item}")
    
    df = df.set_index(['item_id', 'timestamp']).sort_index()
    df = df.drop(columns=['row_index'])

    return df


In [ ]:
autotrain(train_df, test_df)

In [ ]:
test_df = classifier_probability_for_autogluon(train_df)
train_df = classifier_probability_for_autogluon(train_df)

In [ ]:
print(test_df)

In [ ]:
train_df

In [ ]:
counts = test_df['prediction'].value_counts()
print(counts)

counts = train_df['prediction'].value_counts()
print(counts)

In [ ]:
# train_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/train_df_autogluon.parquet')
# test_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/test_df_autogluon.parquet')

#### CDF or Regressor

### xPatch

# Darts model

### Dataset

In [ ]:
train_df['date'] = pd.to_datetime(train_df[['Year', 'Month', 'Day', 'Hour']])
train_df = train_df.set_index('date')
train_df = train_df.drop(columns=['Year', 'Month', 'Day', 'Hour', 'index'])

In [ ]:
list_target_series = []
list_past_covariates = []

unique_stations = train_df['Station'].unique()

for station_id in unique_stations:
    station_df = train_df[train_df['Station'] == station_id]

    # Drop the 'Station' column as it's no longer needed within a single TimeSeries
    station_df = station_df.drop(columns=['Station'])

    # Create TimeSeries for the current station
    target_s = TimeSeries.from_dataframe(station_df, value_cols=['Rain'])
    past_covs_s = TimeSeries.from_dataframe(station_df, value_cols=['Temperature', 'Pressure', 'Humidity', 'GSMap'])

    list_target_series.append(target_s)
    list_past_covariates.append(past_covs_s)

print(target_s[0])

In [ ]:
dataset = PastCovariatesSequentialDataset(
    target_series=list_target_series,    # Pass the list of target TimeSeries
    covariates=list_past_covariates,     # Pass the list of past_covariates TimeSeries
    input_chunk_length=input_chunk_length,
    output_chunk_length=output_chunk_length
)

### Model

In [ ]:
model = RegressionEnsembleModel(
    forecasting_models = [
        NaiveSeasonal(K=12),
        LinearRegressionModel(lags=4)
    ],
    regression_train_n_points=20
)

# GNN Model

In [2]:
rain_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/HII_station_2015-2020.csv")
gsmap_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/GSMap_now_station_2015-2020.csv")
humid_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Humidity_2015-2020.csv")
pressure_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Pressure_2015-2020.csv")
temp_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Temperature_2015-2020.csv")

coordinate_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Coor_HII_495sta.csv")

In [4]:
coor_rad = coordinate_df[['lat', 'long']]* np.pi / 180 #Convert to radians

distance_threshold = 100  # Define a distance threshold for km

distance_matrix = haversine_distances(coor_rad[['lat', 'long']]) #compute distance matrix
distance_matrix = distance_matrix * 6371

i, j = np.where((distance_matrix < distance_threshold) & (distance_matrix != 0))
edge_index = np.column_stack((i, j))
edge_index = torch.tensor(edge_index, dtype=torch.long).t()

pd.DataFrame(distance_matrix, index=coordinate_df['Station'], columns=coordinate_df['Station'])

In [24]:
class DailyGraphDataset(Dataset):
    def __init__(self, hll_df, gsmap_df , humid_df, pressure_df, temp_df, edge_index):
        self.hll_df = hll_df
        self.humid_df = humid_df
        self.gsmap_df = gsmap_df
        self.pressure_df = pressure_df
        self.temp_df = temp_df
        self.edge_index = edge_index

    def __len__(self):
        return len(self.gsmap_df)

    def __getitem__(self, idx):
        # Extract all features for this day (row `idx`) across all stations (columns 4:).
        gsmap_feat = self.gsmap_df.iloc[idx, 4:].values
        humid_feat = self.humid_df.iloc[idx, 4:].values
        pressure_feat = self.pressure_df.iloc[idx, 4:].values
        temp_feat = self.temp_df.iloc[idx, 4:].values

        # Stack or concatenate the features across feature dimension
        combined = np.stack([gsmap_feat, humid_feat, pressure_feat, temp_feat], axis=1)  # shape: [num_stations, num_features]
        x = torch.tensor(combined, dtype=torch.float)

        # Target rainfall
        y = torch.tensor(self.hll_df.iloc[idx, 4:].values, dtype=torch.float).unsqueeze(1)

        return Data(x=x, y=y, edge_index=self.edge_index.contiguous())

train_ds = DailyGraphDataset(rain_df, gsmap_df, humid_df, pressure_df, temp_df, edge_index)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

In [28]:
train_dl

In [29]:
print(len(train_dl))
# batch = next(iter(train_dl))
batch = train_ds[0]
print(batch)
print(batch.num_graphs)
print(batch.num_nodes)
print(batch.num_edges)
print(batch.is_undirected())
print(batch.num_node_features)
print(batch.num_edge_features)
print(batch.has_isolated_nodes())
print(batch.has_self_loops())
print(batch.is_directed())

In [23]:
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GCNConv(train_ds.num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, train_ds.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)

In [ ]:
model = GCN(hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train():
      model.train()
      optimizer.zero_grad()  # Clear gradients.
      out = model(data.x, data.edge_index)  # Perform a single forward pass.
      loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
      loss.backward()  # Derive gradients.
      optimizer.step()  # Update parameters based on gradients.
      return loss

def test():
      model.eval()
      out = model(data.x, data.edge_index)
      pred = out.argmax(dim=1)  # Use the class with highest probability.
      test_correct = pred[data.test_mask] == data.y[data.test_mask]  # Check against ground-truth labels.
      test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  # Derive ratio of correct predictions.
      return test_acc


for epoch in range(1, 101):
    loss = train()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

# Predict

#### Eval metric